# Lab 2 Part 2: checkpointed LoRA

The first code cell creates a pinned environment and prints its commit, versions,
device and seed. It does not replace packages inside the running notebook kernel.
Windows, Linux, Colab and Kaggle paths are handled without a hard-coded `/content`
directory on a PC. First-time setup needs internet and several GB of free disk.

Use a fresh kernel and Run All. The default is a full run. `LAB2_SMOKE_ONLY=1`
is only for developer checks and is clearly printed; those outputs are not a
completed GPU experiment. Keep your real executed copy after the full run.

RTX 4050 profile: batch 1, accumulation 4, fp16 frozen UNet, gradient checkpointing, cached latents and text conditioning. No bitsandbytes or xformers dependency. The lab grading target remains a 16 GB GPU.

In [1]:
from pathlib import Path
import os, subprocess, sys, platform, json, time

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"
roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in roots if (p / "tools/setup_runtime.py").exists()), None)
if ROOT is None:
    if os.environ.get("LAB2_ROOT"):
        ROOT = Path(os.environ["LAB2_ROOT"]).expanduser().resolve()
    elif Path("/content").exists():
        ROOT = Path("/content/Lab-2")
    elif Path("/kaggle/working").exists():
        ROOT = Path("/kaggle/working/Lab-2")
    else:
        ROOT = Path.home() / "DATA266" / "Lab-2"
    if not ROOT.exists():
        ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/setup_runtime.py").exists():
        raise RuntimeError(f"{ROOT} is an older/incomplete checkout. Update it with git pull, then restart this notebook.")
print("Notebook host:", platform.python_version(), "repository:", ROOT)
print("Mode:", "SMOKE ONLY: no full training or model-backed evaluation" if SMOKE_ONLY else "FULL RUN")
subprocess.run([sys.executable, str(ROOT / "tools/setup_runtime.py"),
                "--device", "cpu" if SMOKE_ONLY else os.environ.get("LAB2_DEVICE", "cuda")], check=True)
PYTHON = Path(json.loads((ROOT / ".lab2_runtime.json").read_text())["python"])
SEED = 266
print("All remaining commands use:", PYTHON)

def run_script(relative, *arguments):
    command = [str(PYTHON), "-u", str(ROOT / relative), *map(str, arguments)]
    logs = ROOT / "runtime_checks"
    logs.mkdir(exist_ok=True)
    log = logs / (Path(relative).stem + ".log")
    print("Running", Path(relative).name, flush=True)
    with log.open("a", encoding="utf-8") as handle:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in process.stdout:
            print(line, end="", flush=True)
            handle.write(line)
            handle.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f"{Path(relative).name} stopped (exit {code}). Full diagnostic log: {log}")


Notebook host: 3.12.6 repository: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2
Mode: SMOKE ONLY: no full training or model-backed evaluation


{
  "commit": "9205deb",
  "python": "3.12.6",
  "seed": 266,
  "executable": "/Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python",
  "versions": {
    "torch": "2.5.1",
    "torchvision": "0.20.1",
    "diffusers": "0.31.0",
    "transformers": "4.46.2",
    "peft": "0.13.2",
    "datasets": "3.1.0",
    "huggingface-hub": "0.26.2"
  },
  "cuda_available": false,
  "gpu": "cpu"
}
PASS: core imports, matching torchvision kernels and device preflight


LAB2_PYTHON=/Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python
All remaining commands use: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python


## Preflight and real optimizer smoke test
A tiny randomly initialized UNet tests forward/backward, frozen base weights, checkpoint save, adapter export and interrupted-run resume. It does not count as the SD 1.5 training run.

In [2]:
if not SMOKE_ONLY:
    run_script("tools/preflight.py", "--require-cuda")
run_script("part2_diffusion/src/smoke_test.py", *([] if SMOKE_ONLY else ["--require-cuda"]))

Running smoke_test.py


Update 1/2, noise MSE 1.154229


{


  "rank": 2,


  "steps": 1,


  "final_loss": 1.1542287468910217,


... (33 outputs left out)

  "gpu_hours": null,


  "peak_allocated_mib": null


}


PASS: cpu LoRA training, frozen base, save/resume, adapter round-trip and tiny diffusion generation


## Prepare the actual dataset
Corrupt images, duplicates and empty captions are counted. The seed really controls the 90/10 split. Evaluation prompts are frozen from held-out captions before either training run.

In [3]:
P2 = ROOT / "part2_diffusion"
if SMOKE_ONLY:
    print("SKIPPED: full dataset download and conditioning cache (developer smoke mode).")
else:
    run_script("part2_diffusion/src/prepare_data.py", "--output-dir", P2 / "data/processed")
    run_script("part2_diffusion/src/generate_eval.py", "--freeze-prompts")

SKIPPED: full dataset download and conditioning cache (developer smoke mode).


## Train both ranks
Only one UNet is trained at a time. Complete checkpoints are resumed automatically. A finished compatible run is kept. Dataset and configuration mismatches stop rather than mixing experiments.

In [4]:
if SMOKE_ONLY:
    print("SKIPPED: both full SD 1.5 runs. The tiny-model test is not submitted training.")
else:
    for name, rank in [("run1", 4), ("run2", 8)]:
        run_script("part2_diffusion/src/train_lora.py", "--config", P2 / "config.yaml",
                   "--run", name, "--rank", rank, "--resume-latest")

SKIPPED: both full SD 1.5 runs. The tiny-model test is not submitted training.


## Generate and calculate metrics
All 120 images use identical settings. Missing adapters are errors, never silently replaced by the base model. The checkpoint safety checker stays enabled. Inference offloads components instead of loading the whole pipeline into VRAM. KID, CLIP and LPIPS are actually computed; human judgments still need real raters.

In [5]:
if SMOKE_ONLY:
    print("SKIPPED: real image generation, quality metrics and human judgments.")
else:
    run_script("part2_diffusion/src/generate_eval.py", "--config", P2 / "config.yaml")
    run_script("part2_diffusion/src/evaluate.py", "--config", P2 / "config.yaml")
    for name in ["run1", "run2"]:
        print(name, json.loads((P2 / "outputs" / name / "training_metrics.json").read_text()))
    print((P2 / "outputs/evaluation.json").read_text())
    from IPython.display import Image, display
    for index in range(8):
        display(Image(filename=str(P2 / "outputs/generations" / f"comparison_{index:02d}.png")))

SKIPPED: real image generation, quality metrics and human judgments.


## Save your work
Keep this executed notebook, both adapters and the outputs folder. Upload weights and images to your team Hugging Face account, not GitHub. Complete the 25 blinded pairs with at least three raters; the code does not invent their choices.